# Pipeline comparison — raw vs s10 vs s11

- **Question:** does the Sprint 1 cleaning help once the model is tuned?
- `exp3` verdict, **untuned** LGBM (`mcs=5`, no `reg_lambda`): s10 74.31, s11 47.42 vs raw 80.05 → "cleaning hurts"
- Tuned champion reaches 87.84 on raw; s10/s11 never re-checked after regularisation, and never run through the NN or neighbour model
- **Design:** dedup on **raw** configuration identity → identical 53,947 train rows for every pipeline, only the feature matrix varies
- Targets are **consensus** throughout; s10's *relabelled* targets are a separate question
- **s11 is champion-only:** `wd512` and `nndelta` cast `X` to `uint8`, silently zeroing the 45 float SVD components
- **s10 × nndelta is not apples-to-apples:** s10 drops 11 CRM columns, so pairs differing by one pack can collapse and contradict
- Needs `data/train_clean.csv` (s10), `data/train_clean_v2.csv` + `data/test_clean_v2.csv` (s11); absent pipelines are skipped and reported
- Notebook form of `scripts/exp6_pipeline_comparison.py`

## Setup

In [ ]:
import os
import sys
import time

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from scipy.stats import binomtest

from src import data, experiment as E, validation as V
from src.models import BinaryRelevanceLGBM
from src.models_nndelta import NeighbourDelta
from src.models_widedeep import WideDeepMultiLabel

d = E.setup(seed=42, val_size=0.2)
AVAILABLE = data.cached_pipelines(d)
print('pipelines this cache can serve:', AVAILABLE)
for p in ('s10', 's11'):
    if p not in AVAILABLE:
        print(f"  {p} unavailable -- needs {list(data.PIPELINE_SOURCES[p])} in data/")

## Configuration

- `LOG_TO_TRACKER` off — re-running would append duplicates to the shared `results.jsonl`; use `scripts/exp6_pipeline_comparison.py` to log
- Probability cache is **float32**, not float16: `nndelta` averages ≤5 neighbours, so values land exactly on the 0.5 threshold and float16 rounding flips predictions

In [ ]:
SEED = 42
THR = 0.5
N_BOOT = 2000
PIPELINES = [p for p in ('raw', 's10', 's11') if p in AVAILABLE]
LOG_TO_TRACKER = False
CACHE_DIR = os.path.join(data.DATA_DIR, 'cache')

CHAMP_PARAMS = dict(min_child_samples=20, reg_lambda=1.0)
WD_PARAMS = dict(hidden=(512,), dropout=0.2, wide=True, lr=2e-3,
                 batch_size=256, max_epochs=80, patience=8)
NND_PARAMS = dict(min_pairs=2, min_rate=0.5, max_nb=5)

# model -> pipelines it can legitimately consume (see the s11 caveat above)
CAN_USE = {'champ': ('raw', 's10', 's11'),
           'wd512': ('raw', 's10'),
           'nndelta': ('raw', 's10')}
FAMILY = {'champ': 'BR LightGBM (variance reduction)',
          'wd512': 'Wide & Deep multi-label NN',
          'nndelta': 'Neighbour + pack-delta',
          'blend': 'avg(wd512, nndelta, BR mcs20)'}


def make(name):
    if name == 'champ':
        return BinaryRelevanceLGBM(CHAMP_PARAMS)
    if name == 'wd512':
        return WideDeepMultiLabel(seed=SEED, verbose=False, **WD_PARAMS)
    return NeighbourDelta(**NND_PARAMS)


print('will run:', PIPELINES)

## Shared training rows

- One dedup on raw configuration identity, reused by every pipeline — makes this a feature-set ablation, not three unrelated experiments

In [ ]:
tr, va = d['tr'], d['va']
X_raw = data.features(d, 'raw')
_, Y_tr, _ = E.dedup_configs(X_raw[tr], d['Y_cons'][tr])
codes, _ = V.factorize_rows(X_raw[tr])
_, first = np.unique(codes, return_index=True)
Y_va = d['Y_cons'][va]

# configurations are the independent unit for the paired tests; size = rows per config
cfg, cfg_first, cfg_size = np.unique(d['groups'][va], return_index=True, return_counts=True)
w = cfg_size.astype(np.float64)

print(f'train rows {len(first):,} (identical across pipelines) | Y_tr {Y_tr.shape}')
print(f'val {len(va):,} rows / {len(cfg):,} unique configurations')
pd.DataFrame({'pipeline': PIPELINES,
              'n_features': [data.features(d, p).shape[1] for p in PIPELINES],
              'dtype': [str(data.features(d, p).dtype) for p in PIPELINES]})

## Fit and score

- Budget per pipeline: ~7 min champion (731 LightGBM models), ~2 min `wd512`, <1 min `nndelta`
- Cached probabilities make re-execution instant

In [ ]:
def fit_or_load(name, pipe, X_tr_p, X_va_p):
    f = os.path.join(CACHE_DIR, f'proba_{name}_{pipe}_s{SEED}.npy')
    if os.path.exists(f):
        print(f'    [{name}] cached', flush=True)
        return np.load(f), 0.0
    with E.Timer() as t:
        P = np.asarray(make(name).fit(X_tr_p, Y_tr).predict_proba(X_va_p), dtype=np.float32)
    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(f, P)
    print(f'    [{name}] fitted in {t.s:.0f}s', flush=True)
    return P, t.s


P, ok, rows = {}, {}, []
for pipe in PIPELINES:
    X = data.features(d, pipe)
    X_tr_p, X_va_p = X[tr][first], X[va]
    print(f'{pipe}: {X_tr_p.shape[1]} features, dtype {X_tr_p.dtype}', flush=True)
    done = []
    for name, allowed in CAN_USE.items():
        if pipe not in allowed:
            print(f'    [{name}] SKIPPED on {pipe} -- would silently truncate float features')
            continue
        P[name, pipe], secs = fit_or_load(name, pipe, X_tr_p, X_va_p)
        done.append((name, secs))
    if len(done) == 3:
        P['blend', pipe] = sum(P[n, pipe] for n, _ in done) / 3.0
        done.append(('blend', 0.0))

    for name, secs in done:
        pred = (P[name, pipe] >= THR).astype(np.uint8)
        res = E.score(d, pred)
        ok[name, pipe] = ((pred != Y_va).sum(1) == 0)[cfg_first]
        rows.append(dict(model=name, pipeline=pipe, n_features=X_tr_p.shape[1], **res))
        print(f'      {name:<8} EMR {res["emr"]:.5f}  HL {res["hamming_loss"]:.6f}', flush=True)
        if LOG_TO_TRACKER:
            E.log(f'andre-{name}-{pipe}', f'{pipe}+consensus', FAMILY[name],
                  dict(thr=THR, split_seed=SEED), res, secs,
                  notes=f'{pipe} features, unique configs, consensus labels, unweighted')

## Results — EMR by feature set

In [ ]:
tbl = pd.DataFrame(rows)
emr = (tbl.pivot(index='model', columns='pipeline', values='emr')
       .reindex([m for m in ('champ', 'wd512', 'nndelta', 'blend') if m in set(tbl.model)]))
emr = emr[[p for p in PIPELINES if p in emr.columns]]
if 'raw' in emr.columns:
    for p in [c for c in emr.columns if c != 'raw']:
        emr[f'{p} - raw (pts)'] = (emr[p] - emr['raw']) * 100
display(emr.style.format(lambda v: '-' if pd.isna(v) else
                         (f'{v:+.2f}' if abs(v) < 5 else f'{v:.4%}')))

display(tbl[['model', 'pipeline', 'n_features', 'emr', 'hamming_loss',
             'share_dist1', 'share_dist3plus']].round(5))

## Paired tests vs the same model on raw

- Unit is the unique CRM configuration, not rows — all rows of a configuration get the same prediction
- CI is a cluster (Poisson) bootstrap on the row-weighted EMR difference, the metric the leaderboard reports
- A CI excluding zero is the claim; McNemar p alone has disagreed with it on this data

In [ ]:
rng = np.random.default_rng(0)
sig = []
for (name, pipe), o in ok.items():
    if pipe == 'raw' or (name, 'raw') not in ok:
        continue
    ref = ok[name, 'raw']
    b, c = int((o & ~ref).sum()), int((~o & ref).sum())
    diff = float((w * (o.astype(float) - ref)).sum() / w.sum())
    dv = (o.astype(float) - ref) * w
    boots = []
    for _ in range(0, N_BOOT, 200):
        pw = rng.poisson(1.0, size=(200, len(w)))
        boots.extend((pw @ dv) / (pw @ w))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    sig.append(dict(model=name, comparison=f'{pipe} - raw', dEMR_pts=diff * 100,
                    ci_lo=lo * 100, ci_hi=hi * 100, b=b, c=c,
                    mcnemar_p=binomtest(b, b + c, 0.5).pvalue if b + c else 1.0,
                    significant='yes' if (lo > 0 or hi < 0) else 'no'))

sg = pd.DataFrame(sig)
display(sg.style.format({'dEMR_pts': '{:+.3f}', 'ci_lo': '{:+.3f}', 'ci_hi': '{:+.3f}',
                         'mcnemar_p': '{:.3g}'}))

out = os.path.join(data.DERIVED_DIR, 'sprint2_pipeline_comparison.csv')
tbl.to_csv(out, index=False)
sg.to_csv(os.path.join(data.DERIVED_DIR, 'sprint2_pipeline_significance.csv'), index=False)
print('written ->', out)

## Notes

- `raw` is the incumbent: a pipeline earns a switch only if its CI against `raw` excludes zero and sits above it, for the same model
- A blank s11 cell is a dtype limitation in `wd512`/`nndelta`, not evidence about the SVD features
- Models covered: `champ`, `wd512`, `nndelta`, blend. Pedro's `E-TierB-Chain-LR` (86.81%) is **not** included — it lives in `notebooks/sprint2_tierB.ipynb` on `origin/PedroCorreia`, not in `src/`